<a href="https://colab.research.google.com/github/yshzjq/BeatShaberProject/blob/main/%EB%A8%B8%EC%8B%A0%EB%9F%AC%EB%8B%9D_%EC%8B%AC%ED%99%94/%5B2%EC%9E%A5_2%EA%B0%95%5D_%EC%8B%A4%EC%8A%B5_%EB%B6%80%EC%8A%A4%ED%8C%85_%EA%B3%84%EC%97%B4_%EB%AA%A8%EB%8D%B8%EC%9D%98_%EC%9B%90%EB%A6%AC%EC%99%80_%EB%B0%9C%EC%A0%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.datasets import load_wine
from sklearn.ensemble import (
    AdaBoostClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

SEED = 42
# Wine은 외부 다운로드가 없는 178×13 실제 데이터이며 class 0을 양성 1로 재정의합니다.
wine = load_wine(as_frame=True)
X = wine.data.copy()
y = (wine.target == 0).astype(int)

# 구간 경계를 데이터 분위수로 다시 계산하면 validation·test가 경계에 영향을 줄 수 있습니다.
# 고정 경계와 category dtype을 사용해 숫자 코드가 연속형 크기로 오해되지 않게 합니다.
X["alcohol_band"] = pd.cut(
    X["alcohol"],
    bins=[-np.inf, 12.5, 13.5, np.inf],
    labels=["low", "middle", "high"],
).astype("category")
X["magnesium_band"] = pd.cut(
    X["magnesium"],
    bins=[-np.inf, 88, 105, np.inf],
    labels=["low", "middle", "high"],
).astype("category")

cat_cols = ["alcohol_band", "magnesium_band"]
# 범주형 두 열을 명시하고 나머지를 수치형으로 계산해 같은 열이 두 변환에 중복되지 않게 합니다.
num_cols = [column for column in X.columns if column not in cat_cols]
# test는 모든 CV 후보와 설정을 고정한 뒤 마지막 평가에서만 사용합니다.
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)


def make_pipeline(model):
    """수치형과 범주형을 나눠 변환한 뒤 분류 모델에 연결합니다."""
    # 대치와 one-hot을 Pipeline 안에 두면 각 CV fold의 학습 행에서만 통계와 범주를 학습합니다.
    pre = ColumnTransformer(
        [
            ("num", SimpleImputer(strategy="median"), num_cols),
            (
                "cat",
                Pipeline(
                    [
                        ("impute", SimpleImputer(strategy="most_frequent")),
                        (
                            "onehot",
                            OneHotEncoder(
                                # 새 범주는 0 벡터로 처리해 validation·test 추론이 중단되지 않게 합니다.
                                handle_unknown="ignore", sparse_output=False
                            ),
                        ),
                    ]
                ),
                cat_cols,
            ),
        ],
        sparse_threshold=0.0,
    )
    return Pipeline([("pre", pre), ("model", model)])


def evaluate(name, template):
    """고정된 fold에서 AP·F1·ROC-AUC의 교차검증 결과를 요약합니다."""
    # 세 지표는 같은 예측과 fold에서 계산되며 모델 선택의 1차 기준은 mean_AP입니다.
    out = cross_validate(
        template,
        X_dev,
        y_dev,
        cv=cv,
        scoring={"AP": "average_precision", "F1": "f1", "ROC_AUC": "roc_auc"},
        n_jobs=1,
    )
    return {
        "candidate": name,
        "mean_AP": out["test_AP"].mean(),
        # std_AP는 다섯 fold 점수의 표준편차이며 평균의 표준오차가 아닙니다.
        "std_AP": out["test_AP"].std(ddof=1),
        "mean_F1": out["test_F1"].mean(),
        "mean_ROC_AUC": out["test_ROC_AUC"].mean(),
    }


assert X.shape == (178, 15)
# shape·열 집합·분할 교집합 검사는 범주 파생 열과 데이터 경계가 의도대로인지 확인합니다.
assert not set(cat_cols) & set(num_cols)
assert set(X_dev.index).isdisjoint(set(X_test.index))

In [6]:
templates = {
    # 세 후보는 배깅과 두 부스팅 방식을 같은 전처리·fold에서 비교하도록 구성합니다.

    # Random Forest
    "random_forest": make_pipeline(

        # RandomForestClassifier 여러 개의 결정트리(Decision Tree)를 만들고 결과를 종합하는 모델
        RandomForestClassifier(

            # 결정트리(나무) 개수
            n_estimators=250,

            #클래스 개수가 불균형할 때, 적은 클래스의 중요도를 자동 높임
            class_weight="balanced",

            # CPU 작업을 1개 프로세스로 실행
            n_jobs=1,
            random_state=SEED,
        )


    ),

    # AdaBoost는 Boosting 방식
    "adaboost": make_pipeline(
        # 낮은 learning rate와 여러 약한 학습기를 결합해 한 단계의 과도한 수정을 줄입니다.
        AdaBoostClassifier(
            # 약한 학습기를 150개 결합하겠다는 의미
            n_estimators=150,
            # 각 단계에서 이전 단계의 영향을 얼마나 강하게 반영할지를 조절
            learning_rate=0.05,
            random_state=SEED
        )
    ),

    # Boosting
    "gradient_boosting": make_pipeline(
        GradientBoostingClassifier(
            n_estimators=150,
            learning_rate=0.05,
            random_state=SEED
        )
    ),
}

# 3개 모델 실행
model_table = pd.DataFrame(
    # evaluate()가 동일한 cv와 지표를 사용하므로 mean_AP 차이를 공정하게 읽을 수 있습니다.
    [evaluate(name, template) for name, template in templates.items()]
).sort_values("mean_AP", ascending=False) # mean_AP 점수 기준으로 순위 정하기

# 사람이 적어 둔 dtype 설명만 믿지 않고 모델 입력 직전의 이름을 검사합니다.
# probe의 fit은 열 이름 확인용이며 test 점수나 모델 선택에는 사용하지 않습니다.
probe = templates["gradient_boosting"].fit(X_dev, y_dev)
feature_names = probe.named_steps["pre"].get_feature_names_out()
assert any("alcohol_band" in name for name in feature_names)
assert any("magnesium_band" in name for name in feature_names)
# 지표 범위와 변환 열 assertion이 통과해야 아래 순위표를 해석할 수 있습니다.
assert model_table["mean_AP"].between(0, 1).all()
print(model_table.round(4))
print("categorical features encoded: PASS")

           candidate  mean_AP  std_AP  mean_F1  mean_ROC_AUC
0      random_forest   1.0000  0.0000   0.9882        1.0000
1           adaboost   0.9978  0.0050   0.9787        0.9988
2  gradient_boosting   0.9105  0.0649   0.9101        0.9531
categorical features encoded: PASS


In [7]:
config_rows = []

# 3번 반복
for learning_rate, n_estimators in [(0.1, 80), (0.05, 160), (0.02, 300)]:
    # 이름에 두 설정을 포함해 표의 행과 templates 사전의 객체가 같은 후보를 가리키게 합니다.
    name = f"gb_lr{learning_rate}_n{n_estimators}"

    # 전처리 + Gradient Boosting이 합쳐진 전체 모델
    template = make_pipeline(
        GradientBoostingClassifier(
            learning_rate=learning_rate,
            n_estimators=n_estimators,
            random_state=SEED,
        )
    )


    # 심화 문제의 최종 선택에서 같은 CV 후보 객체를 그대로 찾습니다.
    # evaluate()를 재사용하므로 세 조합 모두 기존 모델과 동일한 5-fold AP 계약을 따릅니다.
    templates[name] = template
    row = evaluate(name, template)
    row.update(
        {"learning_rate": learning_rate, "n_estimators": n_estimators}
    )
    config_rows.append(row)

# mean_AP 가 높은 모델 순으로 정렬
config_table = pd.DataFrame(config_rows).sort_values("mean_AP", ascending=False)
# 설정 열과 지표 범위를 함께 검사하면 표 작성 중 값이 빠지는 오류를 찾을 수 있습니다.
assert config_table["mean_AP"].between(0, 1).all()
assert set(config_table["n_estimators"]) == {80, 160, 300}
print(config_table.round(4))

        candidate  mean_AP  std_AP  mean_F1  mean_ROC_AUC  learning_rate  \
0    gb_lr0.1_n80   0.9107  0.0650   0.9101        0.9537           0.10   
2  gb_lr0.02_n300   0.9107  0.0650   0.9101        0.9463           0.02   
1  gb_lr0.05_n160   0.9105  0.0649   0.9101        0.9531           0.05   

   n_estimators  
0            80  
2           300  
1           160  


In [9]:

# 데이터를 나눈다
# Early Stopping을 확인하기 위해서 학습시키다가 성능이 더 이상 종아지지 않으면 중간에 학습을
# 멈추는 기능
# X_dev -> 80%의 X_fit, 20%의 X_valid
# y_dev -> 80%의 y_fit, 20%의 y_valid
X_fit, X_valid, y_fit, y_valid = train_test_split(
    X_dev, y_dev, test_size=0.2, stratify=y_dev, random_state=SEED
)
# 이 분할은 early stopping 동작을 보는 진단용이며 봉인된 test와 겹치지 않습니다.
early = make_pipeline(
    # HistGradientBoostingClassifier_ Gradient Boosting 계열의 분류 모델
    # 여러 개의 결정 트리(Decision Tree)를 순서대로 만들어 이전 모델의 틀린 부분을 계속 보완하는 모델
    HistGradientBoostingClassifier(

        # 최대 500번까지 학습
        max_iter=500,

        learning_rate=0.05,

        # 학습하면서 성능이 안좋아 지면 중지
        early_stopping=True,

        # validation 데이터의 일부를 따로 떼어 사용
        validation_fraction=0.15,

        # 성능이 20번 연속으로 좋아지지 않으면 멈춤
        n_iter_no_change=20,
        random_state=SEED,
    )
).fit(X_fit, y_fit) # 실제 모델 학습


# n_iter_가 max_iter보다 작다면 내부 validation에서 개선이 멈춰 조기 종료된 것입니다.

# X_valid 데이터를 넣어 양성일 확률
valid_prob = early.predict_proba(X_valid)[:, 1]
early_result = {

    # .n_iter_ 실제로 몇 번 학습하고 멈췄는지
    "n_iter": early.named_steps["model"].n_iter_,

    # 모델이 양성 확률을 얼마나 잘 순위화했는지 평가
    "AP": average_precision_score(y_valid, valid_prob),

    #  valid_prob >= 0.5 , 확률을 실제 클래스 0/1로 바꾸는 것
    "F1": f1_score(y_valid, valid_prob >= 0.5),

    # 양성과 음성을 얼마나 잘 구분하는지 평가
    "ROC_AUC": roc_auc_score(y_valid, valid_prob),
}
assert 1 <= early_result["n_iter"] <= 500
# 세 진단 지표는 CV 순위표에 합치지 않고 early stopping 결과를 읽는 데만 사용합니다.
print("early stopping:", early_result)

# 최종 순위에는 오직 같은 5-fold CV로 평가한 후보만 포함합니다.
# 원본 세 후보와 boosting 설정 후보가 모두 같은 mean_AP 정의를 공유합니다.

# model_table, config_table 둘을 합침
all_cv = pd.concat([model_table, config_table], ignore_index=True)

# mean_AP 기준으로 가장 좋은 모델 찾기
selected_name = str(all_cv.sort_values("mean_AP", ascending=False).iloc[0].candidate)

# 모델 이름 저장
selected_template = templates[selected_name]

# 선택을 끝낸 뒤 개발 데이터 전체에 적합하고 test를 한 번만 평가합니다.
# 선택한 모델 다시 평가
final_model = selected_template.fit(X_dev, y_dev)
test_prob = final_model.predict_proba(X_test)[:, 1]
final_test = {
    # AP와 ROC-AUC는 확률 순위, F1은 고정 0.5 임계값 결과를 보여 줍니다.
    "AP": average_precision_score(y_test, test_prob),
    "F1": f1_score(y_test, test_prob >= 0.5),
    "ROC_AUC": roc_auc_score(y_test, test_prob),
}
# assertion은 선택 이름·최고 행 일치와 최종 지표 범위를 확인하며 test로 재선택하지 않습니다.
assert selected_name == all_cv.loc[all_cv["mean_AP"].idxmax(), "candidate"]
assert all(0 <= value <= 1 for value in final_test.values())
print("selected:", selected_name)
print("final test:", final_test)